# 07 — Multi-Agent + A2A

So far our system is **vertical**: one agent, many tools (via MCP). Now we go **horizontal**: multiple specialized agents talking to each other.

**Two protocols, two jobs:**
- **MCP** = vertical. App ↔ tool. "How does my agent reach a database?"
- **A2A** = horizontal. Agent ↔ agent. "How does my coordinator delegate to a research specialist?"

They're complementary, not competitive. A real multi-agent system uses both: A2A for delegation, MCP for each agent to access its private tool stack.

**By the end of this notebook you will:**
1. Read an **AgentCard** at `/.well-known/agent-card.json` — A2A's discovery primitive
2. Send a task via `tasks/send` JSON-RPC and get a result
3. Run **two researcher agents** as A2A servers (each with their own ReAct loop and MCP tools)
4. Run a **Coordinator** that decomposes a topic, fans out via A2A, synthesizes a brief

Lecture reference: **S7 §7** (A2A protocol).


## 1. The architecture we're building

```
                                  Coordinator
                          (decomposes + synthesizes)
                                       │
             ┌─────────────────────────┼─────────────────────────┐
             │                         │                         │
          A2A                       A2A                       A2A
  tasks/send {q1}            tasks/send {q2}           tasks/send {q3}
             │                         │                         │
             ▼                         ▼                         ▼
       Researcher #1             Researcher #2             Researcher #3
      (ReAct + MCP)             (ReAct + MCP)             (ReAct + MCP)
             │                         │                         │
          web_search +              same                      same
          fetch_url
```

Each researcher is a fully independent process with its own LLM, its own tool registry, its own ReAct loop. They could be on different machines, written in different languages, owned by different teams. **A2A is what makes them composable.**

## 2. The AgentCard

Every A2A agent publishes a JSON document at a well-known URL. Read ours:

In [ ]:
from deepbrief.a2a.agent_card import researcher_card
import json

card = researcher_card(public_url="http://localhost:9001")
print(json.dumps(card.model_dump(), indent=2))

Think of this as the **digital business card** for an agent: name, version, what skills it has, what auth it expects, what input/output formats it accepts. An agent registry can crawl `/.well-known/agent-card.json` across an organization and build a directory by skill. **Service Discovery for the agent world.**


## 3. Spin up two researchers

We'll run two researcher servers on ports 9001 and 9002. Each is the same code but a different process — that's the whole point. In production they could be different vendors / frameworks.

In [ ]:
import os, subprocess, sys, time

subprocess.run(["pkill", "-f", "deepbrief.agents.researcher"], capture_output=True)
time.sleep(1)

researchers = []
for port in [9001, 9002]:
    proc = subprocess.Popen(
        [sys.executable, "-m", "deepbrief.agents.researcher", "--port", str(port)],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE,
        env=os.environ.copy(),
    )
    researchers.append(proc)
    print(f"Researcher on :{port}  pid={proc.pid}")

time.sleep(3)   # FastAPI/uvicorn startup is ~1-2s per process
for p, port in zip(researchers, [9001, 9002]):
    print(f":{port} alive={p.poll() is None}")

## 4. Discover them by fetching the card

In [ ]:
from deepbrief.a2a.client import fetch_agent_card

for port in [9001, 9002]:
    card = await fetch_agent_card(f"http://localhost:{port}")
    print(f":{port} → {card.name} v{card.version}")
    print(f"   skills: {[s.id for s in card.skills]}")

## 5. Send a task with `tasks/send`

This is the JSON-RPC 2.0 call. The server runs its full ReAct loop (web_search → fetch_url → reason) and returns the final agent message.

In [ ]:
from deepbrief.a2a.client import send_task

reply = await send_task(
    "http://localhost:9001",
    "What is the Model Context Protocol (MCP)? Who created it and when?",
)
print(reply)

Look at the wire form. The JSON-RPC call we just made:

```jsonc
POST http://localhost:9001/a2a
Content-Type: application/json

{
  "jsonrpc": "2.0",
  "id": "<uuid>",
  "method": "tasks/send",
  "params": {
    "message": {
      "role": "user",
      "parts": [{"text": "What is the Model Context Protocol..."}]
    }
  }
}
```

And the response:
```jsonc
{
  "jsonrpc": "2.0", "id": "<same-uuid>",
  "result": {
    "task": {
      "id": "<task-uuid>", "state": "completed",
      "messages": [
        {"role": "user", "parts": [...]},
        {"role": "agent", "parts": [{"text": "...the answer..."}]}
      ]
    }
  }
}
```

We're using **synchronous** `tasks/send` — client waits for the agent to finish. Real A2A also supports streaming (SSE) and push-notification webhooks for long-running tasks (a 30-min research job). Same protocol, different execution mode. We won't implement those here.

## 6. Coordinator: decompose → fan out → synthesize

Read the Coordinator class first:

In [ ]:
import inspect
from deepbrief.agents.coordinator import Coordinator

# Show the methods (not the full prompt strings)
src = inspect.getsource(Coordinator)
print(src)

Notice: **the coordinator does not use ReAct internally.** Its workflow is deterministic — decompose, fan out, synthesize. The only LLM steps are the two creative ones (decomposition and synthesis). This is the **bounded autonomy** pattern from S8 §5: workflow with LLM steps embedded, NOT a free agent. It's predictable, auditable, and ~5× cheaper than letting an agent loop figure out the same plan.


## 7. Run a full research job

Pick a topic and watch the coordinator delegate. Each call to `Coordinator.run()` makes:
- 1 LLM call to decompose
- N parallel A2A calls (one per sub-question)
- 1 LLM call to synthesize

Inside each researcher, the ReAct loop makes its own LLM + tool calls.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

coordinator = Coordinator(researcher_urls=[
    "http://localhost:9001",
    "http://localhost:9002",
])

TOPIC = "State of the Model Context Protocol (MCP) in 2026"
draft = await coordinator.run(TOPIC)

print("=== SUB-QUESTIONS ===")
for i, q in enumerate(draft.subquestions, 1):
    print(f"{i}. {q}")

print("\n=== RESEARCHER NOTES ===")
for i, n in enumerate(draft.notes, 1):
    print(f"\n--- note {i} ---")
    print(n[:400] + ('...' if len(n) > 400 else ''))

print("\n=== SYNTHESIZED BRIEF ===")
print(draft.markdown)

## 8. What this composition gives you

Try this thought experiment: replace researcher 1 with a *different vendor's* agent — say, an agent built with LangGraph instead of our raw OpenAI loop. As long as it speaks A2A (`/.well-known/agent-card.json` + `tasks/send`), the coordinator doesn't need to change. **That's the whole pitch of A2A.** Multi-vendor, multi-framework, swap any node without rewiring.

Compare with calling functions directly: you'd be locked to one process, one language, one framework. Compare with putting everything in MCP: MCP doesn't have task IDs, multi-turn conversation state, or push-callback semantics — it's the wrong tool for agent-to-agent.


## 9. Cleanup

In [ ]:
for proc in researchers:
    proc.terminate()
    try:
        proc.wait(timeout=5)
    except subprocess.TimeoutExpired:
        proc.kill()
print("researchers stopped")

## 10. Self-check

1. MCP solves vertical integration (app ↔ tool). What does A2A solve? When would you reach for one vs the other?
2. What's the role of the AgentCard? Where does it live? Why is `/.well-known/` the convention?
3. The coordinator doesn't use ReAct internally — why is a deterministic workflow the right choice for it?
4. We used synchronous `tasks/send`. When would you switch to streaming or push notifications?
5. If researcher #2 crashes mid-task, what does the coordinator see? How would you make it more robust?

## What's next

Notebook **08** — the **Capstone**. Wire everything together with one final piece: a **Human-in-the-loop editor** that gates the brief before it's saved to disk. This is the S8 §5.5 *Tier 2 sync HITL* pattern — and it's exactly how Deep Research products in production handle the "AI generated this, do you actually want to publish it?" decision.